# Lab 8 trên Google Colab

Dùng notebook này cho **AI và các lệnh phân tích**. CVAT chạy riêng: máy bình thường dùng CVAT local; máy yếu mở CVAT của chương trình trong trình duyệt. **Không chạy CVAT server trong Colab.** Bài nộp của mỗi người là cá nhân; bạn có thể tự kiểm hoặc trao đổi với bạn sau khi đã chốt quyết định độc lập.

Làm từ trên xuống. Sau mỗi mốc khóa, tải `submission.zip` về máy. Phiên Colab có thể bị xóa khi ngắt kết nối. Notebook chạy YOLO thật trên frame demo `d01`; bài 10 frame và dự đoán pool vẫn dùng đầu vào đã đóng băng để mọi học viên so sánh công bằng.


## 1. Đưa repo riêng của bạn vào Colab

Trên GitHub, mở **repo public của chính bạn** → **Code → Download ZIP**. Chạy ô dưới, chọn ZIP vừa tải. Không dán GitHub token hay mật khẩu vào notebook.

In [ ]:
from google.colab import files
from pathlib import Path
import io, os, subprocess, sys, zipfile, shutil
uploaded = files.upload()
assert len(uploaded) == 1 and next(iter(uploaded)).lower().endswith('.zip'), 'Chọn đúng 1 ZIP repo GitHub'
archive_name = next(iter(uploaded))
work = Path('/content/day8')
if work.exists(): shutil.rmtree(work)
work.mkdir()
with zipfile.ZipFile(io.BytesIO(uploaded[archive_name])) as z:
    for info in z.infolist():
        target = (work / info.filename).resolve()
        if not target.is_relative_to(work.resolve()): raise ValueError('ZIP chứa đường dẫn không an toàn')
    z.extractall(work)
candidates = [p for p in work.rglob('lab8') if p.is_dir() and (p.parent/'data'/'SHA256SUMS').exists()]
assert len(candidates) == 1, 'ZIP cần chứa đúng một repo Day 8 có lab8/ và data/'
ROOT = candidates[0].parent
os.chdir(ROOT)
def run(*args):
    result = subprocess.run([sys.executable, '-m', 'lab8', *args], cwd=ROOT, text=True)
    if result.returncode: raise RuntimeError('Lệnh thất bại; đọc dòng LỖI phía trên')
run('verify-data')
print('Repo đang dùng:', ROOT)

## 2. Tự xác định lộ trình và chạy AI thật

Điền **GitHub username**, không điền email hay URL. Cùng một username luôn nhận cùng lộ trình A/B và chi phí S1/S2. Ghi hai giá trị này lại để mở đúng job CVAT.

In [ ]:
HO_TEN = 'Họ Tên' #@param {type:'string'}
GITHUB_USER = 'tai-khoan-github' #@param {type:'string'}
assert HO_TEN.strip() != 'Họ Tên' and GITHUB_USER != 'tai-khoan-github', 'Hãy điền tên và GitHub username thật'
run('init', '--name', HO_TEN, '--github-user', GITHUB_USER)
import json
info = json.loads((ROOT/'submission'/'info.json').read_text())
print('Lộ trình:', info['group'], '| Chi phí:', info['scenario'])


### Thử mô hình trên frame demo `d01`

Ô sau tải YOLO11n về Colab và chạy hai lần với ngưỡng confidence khác nhau. Đây là **thực hành AI thật**, chỉ trên ảnh demo đã công khai. Quan sát box nào biến mất khi tăng ngưỡng, và xe nào vẫn bị bỏ sót. Không dùng kết quả này để thay file pre-label của bài chính.

In [ ]:
%pip -q install ultralytics==8.4.145
from ultralytics import YOLO
from IPython.display import display
from PIL import Image
model = YOLO('yolo11n.pt')
frame = str(ROOT/'data'/'frames'/'lab'/'d01.jpg')
for threshold in (0.25, 0.55):
    result = model.predict(frame, conf=threshold, device='cpu', verbose=False)[0]
    print(f'Ngưỡng {threshold}: {len(result.boxes)} box (nhãn COCO của mô hình)')
    display(Image.fromarray(result.plot()[..., ::-1]))

In [ ]:
QUAN_SAT = 'Ghi một box thay đổi khi tăng ngưỡng và một giới hạn của dự đoán AI' #@param {type:'string'}
assert len(QUAN_SAT.strip()) >= 25 and 'Ghi một box' not in QUAN_SAT, 'Viết quan sát của bạn trước khi chạy ô này'
from datetime import datetime, timezone
(ROOT/'submission'/'ai_probe.json').write_text(json.dumps({'frame':'d01','thresholds':[0.25,0.55],'observation':QUAN_SAT.strip(),'created_at_utc':datetime.now(timezone.utc).isoformat()}, ensure_ascii=False, indent=2))
print('Đã lưu thực hành AI: submission/ai_probe.json (không thay rubric 100 điểm)')

## 3. Làm bài trên CVAT

- **Máy bình thường:** dùng CVAT local theo card CVAT trong repo của bạn, tạo đúng task A/B sau khi đã biết lộ trình.
- **Máy yếu:** mở hai job trên CVAT của chương trình do Lab Coach gửi. Job tay trước, job có AI sau.
- Sau mỗi job, export **CVAT for images 1.1**, bỏ Save images. File tải về thường là ZIP chứa `annotations.xml`.

Ô tiếp theo nhận một ZIP export hoặc XML và đặt vào đúng tên trong `submission/`. Chạy riêng cho job tay, job AI và bản sửa lại. **Không ghi đè `assisted.xml` sau khi đã khóa.**

In [ ]:
TEN_FILE = 'manual_X.xml' #@param ['manual_X.xml','manual_Y.xml','assisted.xml','assisted_rework.xml']
assert TEN_FILE in ({'manual_X.xml' if info['group']=='A' else 'manual_Y.xml', 'assisted.xml', 'assisted_rework.xml'})
up = files.upload()
assert len(up) == 1, 'Chọn đúng một file export'
name, content = next(iter(up.items()))
if name.lower().endswith('.zip'):
    import io
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        names = [n for n in z.namelist() if n.endswith('/annotations.xml') or n == 'annotations.xml']
        assert len(names) == 1, 'ZIP phải có đúng một annotations.xml'
        content = z.read(names[0])
else:
    assert name.lower().endswith('.xml'), 'Chọn ZIP hoặc XML export từ CVAT'
assert content.lstrip().startswith(b'<?xml') or content.lstrip().startswith(b'<annotations'), 'Không phải CVAT XML'
(ROOT/'submission'/TEN_FILE).write_bytes(content)
print('Đã lưu:', TEN_FILE)

## 4. Các mốc phân tích trên Colab

Sau khi đã upload hai bản export: điền `decision_log.csv` (ít nhất 5 dòng của bạn) và `prediction.md`. Để sửa file mẫu, chạy ô tải file, chỉnh trên máy bằng trình soạn thảo/bảng tính, rồi chạy ô upload trả lại. Nhớ đặt đúng tên.

In [ ]:
FILE_CAN_SUA = 'decision_log.csv' #@param {type:'string'}
allowed = {'decision_log.csv','prediction.md','interpretation.md','rework_log.csv','ranking.csv','ranking_rationale.md','peer_check.md','case_review.md','next_round.md','reflection.md'}
downloadable = allowed | {'frame_scores.csv','al_eval.json'}
assert FILE_CAN_SUA in downloadable
files.download(str(ROOT/'submission'/FILE_CAN_SUA))


In [ ]:
up = files.upload()
for name, content in up.items():
    assert name in allowed, f'Tên file không hợp lệ: {name}'
    (ROOT/'submission'/name).write_bytes(content)
    print('Đã cập nhật', name)

Chạy từng ô **đúng thời điểm** theo GUIDE.md trong repo của bạn. Sau `lock`, các file đã khóa phải giữ nguyên. Gửi mã khóa cho Lab Coach để nhận gói reference 1. Sau `lock-ranking`, nhận gói 2. Nếu phiên Colab bị ngắt, tải lại ZIP repo đã chứa `submission/` và chạy lại từ ô 1.

In [ ]:
run('states')
# Tải prediction.md ở ô trên, điền dự đoán rồi upload lại trước khi chạy ô kế tiếp.


In [ ]:
run('lock')
# Gửi mã khóa vừa in cho Lab Coach, sau đó mới lấy gói reference 1.


In [ ]:
up = files.upload()
assert len(up) == 1 and next(iter(up)).lower().endswith('.zip')
name, content = next(iter(up.items()))
pack = Path('/content')/name
pack.write_bytes(content)
run('install-reference', '--zip', str(pack))
run('profile')
print('Đã tạo case_review.md: điền một ca frame C sau khi xem reference. Làm cá nhân hoặc cùng bạn đều hợp lệ.')


In [ ]:
# Điền rework_log.csv và upload assisted_rework.xml trước khi chạy:
run('profile', '--rework')


In [ ]:
run('frame-scores')
import csv
with (ROOT/'submission'/'frame_scores.csv').open(encoding='utf-8') as f:
    suggestions = sorted(csv.DictReader(f), key=lambda r: int(r['ai_priority']))
print('Top 5 gợi ý model:')
for row in suggestions[:5]:
    print(row['frame_id'], 'score=', row['sum_lc'], 'cost=', row['cost'], 'chọn=', row['ai_suggested'])
print('Xem ảnh và phản biện trước khi điền ranking.csv; gợi ý chưa biết ảnh trùng hay vật model bỏ sót.')
# Tải ranking.csv, ranking_rationale.md; điền theo card pool và cost rồi upload lại.


In [ ]:
# Hoàn tất peer_check.md và upload lại trước khi chạy:
run('lock-ranking')
# Gửi mã khóa xếp hạng cho Lab Coach rồi lấy gói 2.


In [ ]:
up = files.upload()
assert len(up) == 1 and next(iter(up)).lower().endswith('.zip')
name, content = next(iter(up.items()))
pack = Path('/content')/name
pack.write_bytes(content)
run('install-reference', '--zip', str(pack))
run('al-eval')
print('Đọc al_eval.json, điền next_round.md và reflection.md, upload lại rồi chạy ô kiểm bài bên dưới.')


### Kết thúc: đề xuất lượt sau và kiểm gói nộp

Điền `case_review.md` sau gói 1: làm cá nhân thì so bản đã khóa với reference; làm cùng bạn thì so hai quyết định độc lập sau khi cả hai đã khóa. Điền `next_round.md` sau `al-eval`: chọn ba frame **chưa chọn** cho lượt giả định tiếp theo. Dùng ô tải/upload ở trên để sửa hai file này. Chưa huấn luyện lại model trong lab này.


In [ ]:
run('check-submission')


## 5. Tải kết quả về máy và push repo riêng

Chạy ô dưới sau mỗi mốc khóa và cuối buổi. ZIP chỉ chứa `submission/`, **không chứa reference**. Trong GitHub Desktop, Clone repository của bạn về máy. Giải nén vào bản clone đó, kiểm tra file trong `submission/`, rồi chọn Commit to main → Push origin lên GitHub repo public của bạn, sau khi đã khóa cả hai phần. Thư mục Download ZIP không push trực tiếp được. Việc chạy `check-submission` chỉ kiểm file; hệ thống sẽ chấm sau khi bạn nộp.

In [ ]:
from google.colab import files
out = Path('/content/submission.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted((ROOT/'submission').rglob('*')):
        if p.is_file(): z.write(p, p.relative_to(ROOT))
print('Đã đóng gói', len(zipfile.ZipFile(out).namelist()), 'file. Tải về và push lên repo riêng.')
files.download(str(out))